# 03 — Pairs Trading (KO / PEP)

Cointegration-style spread trade: rolling OLS hedge ratio, z-score entries.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.backtest.runner import run
from src.data.store import get_or_fetch_panel


In [ ]:
from src.strategies.pairs import Pairs, _rolling_beta

close = get_or_fetch_panel(['KO', 'PEP'], '2010-01-01')
close.tail(3)


In [ ]:
strat = Pairs(window=60, entry_z=2.0, exit_z=0.5, leg_size=0.5)
result = run(close, strat)
print(result.summary())


In [ ]:
pf = result.portfolio
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
pf.value().plot(ax=axes[0]); axes[0].set_title('Equity')
pf.drawdown().plot(ax=axes[1], color='red'); axes[1].set_title('Drawdown')
plt.tight_layout(); plt.show()


### Spread + z-score


In [ ]:
beta = _rolling_beta(close['KO'], close['PEP'], 60)
spread = close['KO'] - beta * close['PEP']
z = (spread - spread.rolling(60).mean()) / spread.rolling(60).std()
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
spread.plot(ax=axes[0]); axes[0].set_title('KO - β·PEP spread')
z.plot(ax=axes[1])
for k, c in [(2, 'red'), (-2, 'red'), (0.5, 'gray'), (-0.5, 'gray')]:
    axes[1].axhline(k, color=c, ls='--', alpha=0.5)
axes[1].set_title('z-score'); plt.tight_layout(); plt.show()


In [ ]:
w = strat.weights(close)
fig, ax = plt.subplots(figsize=(11, 3))
w.iloc[-500:].plot(ax=ax)
ax.set_title('Pair weights (last 500 bars)'); ax.legend(); plt.show()
